# Free Video Transcriber (Ultra-Fast Google Colab Version)

This notebook allows you to use Google Colab's **free GPUs** to transcribe YouTube or Facebook videos incredibly fast without any limits. It uses `faster-whisper` for maximum speed and shows you the transcription in real-time!

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Run the first cell to install the required libraries.
3. Run the second cell, paste your URL, and watch your transcript generate!

In [ ]:
# Install dependencies (Run this first!)
!pip install yt-dlp faster-whisper
!apt update && apt install ffmpeg -y

In [ ]:
# Run the transcriber
import yt_dlp
from faster_whisper import WhisperModel
import os

VIDEO_URL = "" # @param {type:"string"}
MODEL_SIZE = "large-v3" # @param ["tiny", "base", "small", "medium", "large-v3"]

if not VIDEO_URL:
    print("Please enter a valid VIDEO_URL above.")
else:
    print(f"Downloading audio from {VIDEO_URL}...")
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': '%(id)s.%(ext)s',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
        }],
        'quiet': True,
    }
    
    audio_file = None
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(VIDEO_URL, download=True)
            audio_file = f"{info['id']}.m4a"
            
        print(f"Loading Faster-Whisper model '{MODEL_SIZE}' (this uses GPU)...\n")
        # Run on GPU with FP16 for speed
        model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")
        
        print("Transcribing (showing output in real-time):\n")
        segments, info = model.transcribe(audio_file, beam_size=5)
        
        print(f"Detected language '{info.language}' with probability {info.language_probability:.2f}\n")
        print("=== TRANSCRIPT START ===\n")
        
        full_transcript = []
        for segment in segments:
            # Print each segment as it is transcribed
            print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")
            full_transcript.append(segment.text)
            
        print("\n=== TRANSCRIPT END ===\n")
        
        # Save full transcript
        final_text = " ".join(full_transcript)
        with open("transcript.txt", "w", encoding="utf-8") as f:
            f.write(final_text)
            
        print("Transcription saved to 'transcript.txt' in the Files tab on the left.")
        
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        if audio_file and os.path.exists(audio_file):
            os.remove(audio_file)
